# 12 Confidence Triage

Confidence-Based Candidate Triage (4-class)
==============================================
Suggested notebook name: 12_confidence_triage.ipynb (place in notebooks/)

WHY: A hard label hides how confident the model actually was. Ranks every
candidate by prediction confidence and splits into tiers. Already generic
across any number of classes (loops over le.classes_), so the only changes
here vs the old version are the notebook-style paths and an added class
count printout, useful now that there are 4 possible labels instead of 2.

NOTE: uses random_forest.pkl alone, matching what notebook 06 actually
does for final_classification.csv -- NOT the RF+XGB ensemble from
notebook 05 (that ensemble isn't in the real prediction path yet).

In [1]:
import pickle
import pandas as pd

MODEL_PATH = "../models/random_forest.pkl"
LABEL_ENCODER_PATH = "../models/label_encoder.pkl"
FEATURE_COLS_PATH = "../models/feature_cols.pkl"
DATA_PATH = "../outputs/features.csv"
OUTPUT_PATH = "../outputs/confidence_triage.csv"

HIGH_CONF_THRESHOLD = 0.80
LOW_CONF_THRESHOLD = 0.60  # below this -> "needs follow-up"

with open(MODEL_PATH, "rb") as f:
    pipeline = pickle.load(f)
with open(LABEL_ENCODER_PATH, "rb") as f:
    le = pickle.load(f)
with open(FEATURE_COLS_PATH, "rb") as f:
    feature_cols = pickle.load(f)

df = pd.read_csv(DATA_PATH)
df["tic_id"] = df["tic_id"].astype(str).str.replace(".0", "", regex=False)

missing = [c for c in feature_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing feature columns in {DATA_PATH}: {missing}\nAvailable: {list(df.columns)}")

print(f"Model classes: {list(le.classes_)}")

X = df[feature_cols].fillna(0)
probs = pipeline.predict_proba(X)
pred_idx = probs.argmax(axis=1)
pred_labels = le.inverse_transform(pred_idx)
confidence = probs.max(axis=1)


def tier(c):
    if c >= HIGH_CONF_THRESHOLD:
        return "high_confidence"
    elif c >= LOW_CONF_THRESHOLD:
        return "moderate_confidence"
    else:
        return "needs_follow_up"


results = pd.DataFrame({
    "tic_id": df["tic_id"],
    "predicted_label": pred_labels,
    "confidence": confidence.round(3),
    "tier": [tier(c) for c in confidence],
})

# Per-class probability columns -- automatically adapts to however many
# classes le.classes_ contains (currently 4)
for i, cls in enumerate(le.classes_):
    results[f"prob_{cls}"] = probs[:, i].round(3)

results = results.sort_values("confidence", ascending=False)
results.to_csv(OUTPUT_PATH, index=False)

print(results.to_string(index=False))
print(f"\nSaved -> {OUTPUT_PATH}")
print(f"\nTier counts:\n{results['tier'].value_counts().to_string()}")
print(f"\nPredicted label counts:\n{results['predicted_label'].value_counts().to_string()}")

Model classes: ['eclipsing_binary', 'planet']
   tic_id  predicted_label  confidence                tier  prob_eclipsing_binary  prob_planet
261136641           planet       0.735 moderate_confidence                  0.265        0.735
350618622           planet       0.710 moderate_confidence                  0.290        0.710
229742722           planet       0.690 moderate_confidence                  0.310        0.690
261136679           planet       0.660 moderate_confidence                  0.340        0.660
261139167           planet       0.635 moderate_confidence                  0.365        0.635
237201858           planet       0.630 moderate_confidence                  0.370        0.630
279741379 eclipsing_binary       0.605 moderate_confidence                  0.605        0.395
441075486           planet       0.600 moderate_confidence                  0.400        0.600
271893367 eclipsing_binary       0.595     needs_follow_up                  0.595        0.405
1496

c:\Users\Alivia Hossain\Desktop\exoplanet_detection\Exoplanet_detection\.venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
